# 1. Project Overview

**Smart RAG Document Assistant**

This notebook implements the complete Retrieval-Augmented Generation (RAG) pipeline:
1. Extract text from PDF documents.
2. Clean and chunk the text.
3. Generate embeddings.
4. Store in ChromaDB.
5. Retrieve relevant chunks using an Ollama LLM.
6. Evaluate the quality of retrieval and generation.

# 2. Dataset Description

The dataset consists of 5 realistic scientific and historical PDF documents placed in `data/documents/`. If this folder is empty, the pipeline will gracefully handle the empty dataset and skip embedding/generation steps until documents are added.

# 3. Imports

In [ ]:
import os
import re
import glob
import yaml
import pandas as pd
import uuid

from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import chromadb

import sys
sys.path.append(os.path.abspath(os.path.join('..')))
# Import our new reusable core module
from backend.app.services.rag_core import rag_core

print("Libraries imported successfully.")
print(f"RAG Core Config: {rag_core.config}")


# 4. Loading Documents

Load all PDF files from `data/documents/` and extract text page-by-page, keeping metadata (file name, page number, document id).

In [ ]:
DOCUMENTS_DIR = os.path.join("..", "data", "documents")
os.makedirs(DOCUMENTS_DIR, exist_ok=True)

pdf_files = sorted(glob.glob(os.path.join(DOCUMENTS_DIR, "*.pdf")))

raw_pages = []
failed_files = []

for pdf_path in pdf_files:
    file_name = os.path.basename(pdf_path)
    doc_id = str(uuid.uuid5(uuid.NAMESPACE_DNS, file_name))
    try:
        reader = PdfReader(pdf_path)
        for page_num, page in enumerate(reader.pages, start=1):
            text = page.extract_text()
            if text and text.strip():
                raw_pages.append({
                    "text": text,
                    "metadata": {
                        "source": file_name,
                        "page": page_num,
                        "document_id": doc_id
                    }
                })
    except Exception as e:
        failed_files.append((file_name, str(e)))

print(f"Extracted {len(raw_pages)} page(s) total from {len(pdf_files)} PDF(s).")

# 5. Dataset Inspection

Show the number of documents, pages, total extracted text length, and any failed files.

In [ ]:
print("=== Dataset Inspection ===")
print(f"Number of documents : {len(pdf_files)}")
print(f"Number of pages     : {len(raw_pages)}")
print(f"Extracted length    : {sum(len(p['text']) for p in raw_pages)} characters")
if failed_files:
    print(f"Failed files: {failed_files}")

# 6. Cleaning

Clean the text to:
- Remove unnecessary spaces
- Remove duplicated new lines
- Normalize text (without changing the meaning)

In [ ]:
def clean_text(text: str) -> str:
    text = re.sub(r'\n{2,}', '\n', text)
    text = re.sub(r'[^\S\n]+', ' ', text)
    return text.strip()

for page in raw_pages:
    page["text"] = clean_text(page["text"])
    
print("Text cleaning complete.")

# 7. Chunking

We split the text into manageable chunks using the config values.
- **chunk_size**: Large enough to hold a complete paragraph or concept.
- **chunk_overlap**: Ensures that context isn't lost across chunk boundaries.

In [ ]:
CHUNK_SIZE = rag_core.config.get("chunk_size", 700)
CHUNK_OVERLAP = rag_core.config.get("chunk_overlap", 100)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = []
for page in raw_pages:
    splits = text_splitter.split_text(page["text"])
    for i, chunk_text in enumerate(splits):
        chunks.append({
            "text": chunk_text,
            "metadata": {
                "source": page["metadata"]["source"],
                "page": page["metadata"]["page"],
                "document_id": page["metadata"]["document_id"],
                "chunk_id": f"{page['metadata']['document_id']}_p{page['metadata']['page']}_c{i}"
            }
        })

print(f"Created {len(chunks)} chunks from {len(raw_pages)} pages.")

# 8. Embeddings

Using `sentence-transformers` to embed chunks into vectors. We use the model specified in the config.

In [ ]:
MODEL_NAME = rag_core.embedding_model_name
print(f"Loading embedding model: {MODEL_NAME}")
embedding_model = SentenceTransformer(MODEL_NAME)

embeddings = []
if chunks:
    chunk_texts = [c["text"] for c in chunks]
    print(f"Generating embeddings for {len(chunk_texts)} chunks...")
    embeddings_array = embedding_model.encode(chunk_texts, show_progress_bar=True, batch_size=32)
    embeddings = embeddings_array.tolist()
    print("Embeddings generated.")

# 9. Chroma Storage

Initialize the Chroma vector database and store the chunks, embeddings, and metadata. We persist it to `backend/data/vector_store/` so the backend can reuse it without re-embedding. Note: The collection name is from config and we clean it first to avoid duplicates.

In [ ]:
PERSIST_DIR = os.path.abspath(os.path.join("..", "backend", "data", "vector_store"))
os.makedirs(PERSIST_DIR, exist_ok=True)

chroma_client = chromadb.PersistentClient(path=PERSIST_DIR)
COLLECTION_NAME = rag_core.collection_name

try:
    chroma_client.delete_collection(COLLECTION_NAME)
    print(f"Deleted old collection '{COLLECTION_NAME}'")
except Exception:
    pass

collection = chroma_client.create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"}
)

if chunks and embeddings:
    ids = [c["metadata"]["chunk_id"] for c in chunks]
    documents = [c["text"] for c in chunks]
    metadatas = [c["metadata"] for c in chunks]
    
    collection.add(
        ids=ids,
        documents=documents,
        metadatas=metadatas,
        embeddings=embeddings
    )
    print(f"Stored {collection.count()} vectors into Chroma collection '{COLLECTION_NAME}'.")
    
# Reload the collection in our rag_core instance to ensure it's using the fresh DB
rag_core.load_vector_store(persist_dir=PERSIST_DIR)


# 10. Retrieval Testing

We test the core retrieval logic using the newly built `rag_core` module.

In [ ]:
if collection.count() > 0:
    res = rag_core.retrieve("What is machine learning?")
    print(f"Found {len(res['documents'])} chunks for test query.")
    for m in res['metadatas']:
        print(f" - Source: {m['source']} (Page {m['page']})")

# 11. Ollama Generation

Connect to the local Ollama LLM through `rag_core` to ensure strict context adherence.

In [ ]:
print(f"Testing generation via {rag_core.ollama_url} with model {rag_core.ollama_model}")
res = rag_core.rag_query("What are the limits of classical physics?")
print("\nAnswer:\n" + res["answer"])
print("\nSources:\n" + str(res["sources"]))

# 12. Evaluation

Test the full pipeline with 10 questions and record the results into a DataFrame, including failure analysis.
We test normal, difficult, and out-of-domain questions.

In [ ]:
evaluation_questions = [
    # Normal Questions
    "What is quantum mechanics?",
    "What is the main driver of climate change?",
    "Where was Ancient Egypt located?",
    "What is deep learning?",
    "Why is a non-animal source of vitamin B12 needed?",
    
    # Difficult/Complex Questions
    "How does the uncertainty principle differentiate quantum mechanics from classical physics?",
    "What were the Intermediate Periods in Ancient Egypt?",
    
    # Out-of-Domain Questions (Should say 'Information not found')
    "What is the capital of Japan?",
    "Who won the world cup in 2022?",
    "How do you build a nuclear reactor?"
]

results_data = []

print("Running Evaluation...\n")

for q in evaluation_questions:
    res = rag_core.rag_query(q)
    
    # Basic heuristic check for correctness
    is_out_of_domain = q in ["What is the capital of Japan?", "Who won the world cup in 2022?", "How do you build a nuclear reactor?"]
    not_found_str = "information not found" in res['answer'].lower()
    
    if is_out_of_domain and not_found_str:
        correct = "Yes"
        notes = "Successfully rejected OOD question."
    elif not is_out_of_domain and not not_found_str and len(res['sources']) > 0:
        correct = "Yes"
        notes = "Successfully retrieved and answered."
    else:
        correct = "No"
        notes = "Failed retrieval or incorrect hallucination guard."
        
    results_data.append({
        "Question": q,
        "Retrieved Sources": ", ".join(res['sources']) if res['sources'] else "None",
        "Generated Answer": res['answer'],
        "Correct/Incorrect": correct,
        "Retrieval Quality Notes": notes
    })

df_eval = pd.DataFrame(results_data)
display(df_eval)

# 13. Export

Export evaluation results to CSV and verify vector database.

In [ ]:
csv_path = "evaluation_results.csv"
df_eval.to_csv(csv_path, index=False)
print(f"Exported evaluation results to {csv_path}")

print("\nChecking Vector Store Files:")
for file_path in glob.glob(os.path.join(PERSIST_DIR, "**", "*"), recursive=True):
    if os.path.isfile(file_path):
        size = os.path.getsize(file_path)
        print(f" - {os.path.relpath(file_path, PERSIST_DIR)} ({size} bytes)")

print("\nRAG Pipeline generation complete and backend-ready!")